# 📖 Notebook 3: Anonymization Techniques

When you need to share or analyze personal data without exposing individual identities, you need **anonymization**. This notebook covers four practical techniques used at companies like Microsoft, Google, and Apple.

## Learning Objectives

By the end of this notebook, you'll understand:
- **k-Anonymity**: making every record look like at least k-1 others
- **l-Diversity**: ensuring sensitive values are diverse within each group
- **Differential Privacy**: adding mathematical noise so individuals can't be identified
- **Tokenization**: replacing PII with reversible tokens (pseudonymization)
- When to use each technique and their trade-offs

## 🛠️ Setup

```bash
cd enterprise-patterns/privacy-review
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import hashlib
import random
import math
import uuid
from collections import Counter, defaultdict

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🔍 Let's Load Our Data

First, let's load the user data we'll be anonymizing. In a real scenario, this would be a dataset you want to share with analysts or researchers without exposing individual identities.

In [ ]:
def load_user_data():
    """Load user data that we'll practice anonymizing."""
    conn = get_db_connection()
    cursor = conn.cursor(psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT id, first_name, last_name, email, date_of_birth,
               city, state, zip_code, account_status, signup_source
        FROM users
        ORDER BY id
        LIMIT 20
    """)

    users = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return users

users = load_user_data()

print("📋 Original Data (first 5 users) — BEFORE anonymization")
print("=" * 100)
print(f"  {'ID':<4} {'Name':<20} {'Email':<25} {'DOB':<12} {'City':<15} {'State':<6} {'ZIP'}")
print("-" * 100)
for u in users[:5]:
    name = f"{u['first_name']} {u['last_name']}"
    dob = u['date_of_birth'].strftime('%Y-%m-%d') if u['date_of_birth'] else 'N/A'
    print(f"  {u['id']:<4} {name:<20} {u['email']:<25} {dob:<12} {u['city']:<15} {u['state']:<6} {u['zip_code']}")

print(f"\n⚠️  This data contains PII — let's learn how to protect it!")

---
## 🛡️ Technique 1: k-Anonymity

### The Idea

**k-Anonymity** means that for every person in the dataset, there are at least **k-1 other people** who look identical on the quasi-identifier columns (like age, zip code, gender).

**Why?** If you know someone is "34 years old, lives in 98101, male", and only ONE person matches that in the dataset, you've identified them. But if 5 people match (k=5), you can't tell which one they are.

### How It Works

We **generalize** (make less specific) the quasi-identifiers:
- Age 34 → Age range 30-40
- ZIP 98101 → ZIP 981**
- City "Seattle" → State "WA"

We keep generalizing until every combination appears at least k times.

In [ ]:
def generalize_age(dob, level=1):
    """Generalize a date of birth into age ranges.
    Level 1: 5-year ranges (30-34)
    Level 2: 10-year ranges (30-39)
    Level 3: 20-year ranges (20-39)
    """
    if not dob:
        return "unknown"

    from datetime import date
    today = date.today()
    age = today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

    if level == 1:
        bucket = (age // 5) * 5
        return f"{bucket}-{bucket + 4}"
    elif level == 2:
        bucket = (age // 10) * 10
        return f"{bucket}-{bucket + 9}"
    else:
        bucket = (age // 20) * 20
        return f"{bucket}-{bucket + 19}"

def generalize_zip(zip_code, level=1):
    """Generalize a ZIP code by masking digits.
    Level 1: 981** (keep 3 digits)
    Level 2: 98*** (keep 2 digits)
    Level 3: 9**** (keep 1 digit)
    """
    if not zip_code:
        return "*****"

    keep = max(1, 4 - level)
    return zip_code[:keep] + "*" * (len(zip_code) - keep)

def generalize_location(city, state, level=1):
    """Generalize location.
    Level 1: Keep city
    Level 2: State only
    Level 3: Region only
    """
    regions = {
        "WA": "Pacific NW", "OR": "Pacific NW",
        "CA": "West Coast", "NY": "Northeast",
        "TX": "South", "CO": "Mountain",
        "IL": "Midwest", "MA": "Northeast",
        "FL": "Southeast", "GA": "Southeast"
    }

    if level == 1:
        return city
    elif level == 2:
        return state
    else:
        return regions.get(state, "US")

# Demo: show generalization levels
print("📊 Generalization Levels Demo")
print("=" * 65)
u = users[0]

print(f"\n  Original: DOB={u['date_of_birth']}, ZIP={u['zip_code']}, City={u['city']}, State={u['state']}")
for level in [1, 2, 3]:
    age_gen = generalize_age(u['date_of_birth'], level)
    zip_gen = generalize_zip(u['zip_code'], level)
    loc_gen = generalize_location(u['city'], u['state'], level)
    print(f"  Level {level}:  Age={age_gen:<10} ZIP={zip_gen:<8} Location={loc_gen}")

In [ ]:
def apply_k_anonymity(users, k=3):
    """Apply k-anonymity to user data.
    
    Progressively generalizes quasi-identifiers until every combination
    of (age_range, location, zip) appears at least k times.
    """
    # Try increasing generalization levels until k-anonymity is achieved
    for level in range(1, 4):
        anonymized = []
        for u in users:
            anonymized.append({
                "age_range": generalize_age(u["date_of_birth"], level),
                "location": generalize_location(u["city"], u["state"], level),
                "zip": generalize_zip(u["zip_code"], level),
                "account_status": u["account_status"],
                "signup_source": u["signup_source"]
            })

        # Check: does every quasi-identifier combination appear at least k times?
        groups = Counter()
        for a in anonymized:
            key = (a["age_range"], a["location"], a["zip"])
            groups[key] += 1

        min_group = min(groups.values())
        if min_group >= k:
            return anonymized, level, groups

    return anonymized, 3, groups

# Apply k-anonymity with k=3
k = 3
anon_data, gen_level, groups = apply_k_anonymity(users, k=k)

print(f"🛡️ k-Anonymity Applied (k={k}, generalization level={gen_level})")
print("=" * 80)
print(f"  {'Age Range':<12} {'Location':<15} {'ZIP':<8} {'Status':<12} {'Source'}")
print("-" * 80)
for a in anon_data[:10]:
    print(f"  {a['age_range']:<12} {a['location']:<15} {a['zip']:<8} {a['account_status']:<12} {a['signup_source']}")

print(f"\n📊 Group sizes (each combination appears at least {k} times):")
for group_key, count in sorted(groups.items(), key=lambda x: x[1]):
    bar = "█" * min(count, 20)
    print(f"  {str(group_key):<45} {bar} {count}")

print(f"\n✅ Smallest group has {min(groups.values())} members — k={k} anonymity {'achieved! ✅' if min(groups.values()) >= k else 'NOT achieved ❌'}")
print(f"\n💡 Notice: names, emails, SSNs, and exact DOBs are completely removed.")
print(f"   Only generalized quasi-identifiers remain.")

---
## 🌈 Technique 2: l-Diversity

### The Problem with k-Anonymity

k-Anonymity has a weakness: if everyone in a group has the **same sensitive value**, knowing the group reveals the secret.

Example: If all 5 people in the group (age 30-40, Seattle, 981**) have `account_status = 'suspended'`, and you know someone is in that group, you know they're suspended.

### The Fix

**l-Diversity** requires that within each group, the sensitive attribute has at least **l different values**. This prevents the "all the same" attack.

In [ ]:
def check_l_diversity(anonymized_data, quasi_ids, sensitive_attr, l=2):
    """Check if a dataset satisfies l-diversity.
    
    For each group of records sharing the same quasi-identifiers,
    the sensitive attribute must have at least l distinct values.
    """
    # Group records by quasi-identifiers
    groups = defaultdict(list)
    for record in anonymized_data:
        key = tuple(record[q] for q in quasi_ids)
        groups[key].append(record[sensitive_attr])

    # Check each group
    results = []
    for group_key, sensitive_values in groups.items():
        distinct = len(set(sensitive_values))
        satisfies = distinct >= l
        results.append({
            "group": group_key,
            "size": len(sensitive_values),
            "distinct_values": distinct,
            "values": dict(Counter(sensitive_values)),
            "satisfies_l_diversity": satisfies
        })

    return results

# Check l-diversity for account_status within our k-anonymous groups
quasi_ids = ["age_range", "location", "zip"]
sensitive_attr = "account_status"
l = 2

diversity_results = check_l_diversity(anon_data, quasi_ids, sensitive_attr, l=l)

print(f"🌈 l-Diversity Check (l={l}, sensitive attribute: '{sensitive_attr}')")
print("=" * 80)

for r in diversity_results:
    icon = "✅" if r["satisfies_l_diversity"] else "❌"
    print(f"\n  {icon} Group: {r['group']}")
    print(f"     Size: {r['size']} records, {r['distinct_values']} distinct values")
    print(f"     Values: {r['values']}")

all_satisfy = all(r["satisfies_l_diversity"] for r in diversity_results)
print(f"\n{'✅' if all_satisfy else '❌'} Dataset {'satisfies' if all_satisfy else 'does NOT satisfy'} {l}-diversity")

if not all_satisfy:
    print("\n💡 Fix: Groups with insufficient diversity need further generalization")
    print("   or records must be suppressed (removed) from the dataset.")

---
## 🎲 Technique 3: Differential Privacy

### The Idea

**Differential Privacy** takes a completely different approach: instead of modifying the data itself, we **add random noise** to the answers we compute from the data.

The key insight: if adding or removing any single person from the dataset barely changes the output, then the output doesn't reveal anything about that person.

### The Math (Simplified)

We add noise from a **Laplace distribution**. The amount of noise depends on:
- **Sensitivity**: how much one person can change the answer (usually 1 for counting queries)
- **Epsilon (ε)**: the privacy budget — smaller ε = more noise = more privacy

```
noisy_answer = true_answer + Laplace(0, sensitivity/epsilon)
```

This is what Apple uses for keyboard data, Google uses for Chrome metrics, and the US Census uses for population counts.

In [ ]:
def laplace_noise(sensitivity, epsilon):
    """Generate noise from a Laplace distribution.
    
    Args:
        sensitivity: how much one person can change the query result
        epsilon: privacy budget (smaller = more privacy, more noise)
    
    Returns:
        A random noise value
    """
    scale = sensitivity / epsilon
    # Laplace distribution: draw from Uniform, then transform
    u = random.random() - 0.5
    return -scale * math.copysign(1, u) * math.log(1 - 2 * abs(u))

def dp_count(true_count, epsilon=1.0):
    """Return a differentially private count.
    Sensitivity = 1 (adding/removing one person changes count by 1).
    """
    noise = laplace_noise(sensitivity=1, epsilon=epsilon)
    return max(0, round(true_count + noise))

def dp_average(values, epsilon=1.0, value_range=None):
    """Return a differentially private average.
    Sensitivity = range / n (one person can change avg by at most range/n).
    """
    if not values:
        return 0

    n = len(values)
    true_avg = sum(values) / n

    if value_range is None:
        value_range = max(values) - min(values)

    sensitivity = value_range / n
    noise = laplace_noise(sensitivity, epsilon)
    return round(true_avg + noise, 2)

# Demo: compare true vs differentially private answers
print("🎲 Differential Privacy Demo")
print("=" * 70)

# Query: How many users are in each city?
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT city, COUNT(*) FROM users GROUP BY city ORDER BY COUNT(*) DESC")
city_counts = cursor.fetchall()
conn.close()

print("\n📊 Users per City — True vs Differentially Private Counts")
print(f"  {'City':<18} {'True':>6} {'ε=1.0':>8} {'ε=0.5':>8} {'ε=0.1':>8}")
print("-" * 55)

for city, true_count in city_counts:
    dp_1 = dp_count(true_count, epsilon=1.0)
    dp_05 = dp_count(true_count, epsilon=0.5)
    dp_01 = dp_count(true_count, epsilon=0.1)
    print(f"  {city:<18} {true_count:>6} {dp_1:>8} {dp_05:>8} {dp_01:>8}")

print("\n💡 Lower epsilon = more noise = more privacy")
print("   ε=1.0 is moderate privacy, ε=0.1 is very strong privacy")
print("   Notice: with ε=0.1, small counts become very noisy (good for privacy!)")

In [ ]:
# Show how noise decreases with more data (utility improves at scale)
print("📈 Differential Privacy Gets Better with More Data")
print("=" * 60)
print("\nRunning each query 20 times to show noise distribution:\n")

true_count = 50  # Our actual user count
epsilon = 0.5

results = [dp_count(true_count, epsilon) for _ in range(20)]
avg_result = sum(results) / len(results)
error = abs(avg_result - true_count)

print(f"  True count: {true_count}")
print(f"  DP results: {results}")
print(f"  Average of 20 runs: {avg_result:.1f} (error: {error:.1f})")
print(f"\n💡 Key insight: the AVERAGE of many DP queries converges to the truth.")
print(f"   This is why DP works well for aggregate analytics at scale.")
print(f"   Individual queries are noisy, but patterns in large datasets remain visible.")

---
## 🔑 Technique 4: Tokenization (Pseudonymization)

### The Idea

**Tokenization** replaces PII with random tokens, but **keeps a secure mapping** so you can reverse it when needed. This is **pseudonymization** — the data is still technically personal data (because it's reversible), but it's much safer.

### Why Use It?

- **Internal analytics**: Data engineers work with tokens instead of real emails
- **Cross-system linking**: Join data across databases using tokens without exposing PII
- **Breach impact reduction**: If the analytics DB is breached, attacker only gets tokens
- **GDPR compliance**: Pseudonymization is specifically called out as a safeguard in GDPR

We'll store the token-to-PII mapping in **Redis** (separate from the main database).

In [ ]:
class Tokenizer:
    """Tokenization service: replace PII with tokens, store mapping in Redis."""

    def __init__(self, redis_client, namespace="token"):
        self.r = redis_client
        self.namespace = namespace

    def tokenize(self, value, pii_type="generic"):
        """Replace a PII value with a token.
        If the same value was tokenized before, return the same token (idempotent).
        """
        if not value:
            return None

        # Check if already tokenized
        existing = self.r.get(f"{self.namespace}:value_to_token:{pii_type}:{value}")
        if existing:
            return existing

        # Generate a new token
        token = f"TOK-{pii_type.upper()}-{uuid.uuid4().hex[:12]}"

        # Store bidirectional mapping
        pipe = self.r.pipeline()
        pipe.set(f"{self.namespace}:value_to_token:{pii_type}:{value}", token)
        pipe.set(f"{self.namespace}:token_to_value:{token}", value)
        # Track token metadata
        pipe.hset(f"{self.namespace}:meta:{token}", mapping={
            "pii_type": pii_type,
            "created_at": datetime.now().isoformat()
        })
        pipe.execute()

        return token

    def detokenize(self, token):
        """Recover the original value from a token (requires access to Redis)."""
        return self.r.get(f"{self.namespace}:token_to_value:{token}")

    def delete_mapping(self, token):
        """Permanently delete a token mapping (for GDPR deletion requests)."""
        original = self.r.get(f"{self.namespace}:token_to_value:{token}")
        meta = self.r.hgetall(f"{self.namespace}:meta:{token}")

        if original and meta:
            pii_type = meta.get("pii_type", "generic")
            pipe = self.r.pipeline()
            pipe.delete(f"{self.namespace}:value_to_token:{pii_type}:{original}")
            pipe.delete(f"{self.namespace}:token_to_value:{token}")
            pipe.delete(f"{self.namespace}:meta:{token}")
            pipe.execute()
            return True
        return False

from datetime import datetime

# Create the tokenizer
r = get_redis_client()
tokenizer = Tokenizer(r)

# Tokenize user data
print("🔑 Tokenization Demo")
print("=" * 90)
print(f"  {'Original Email':<30} {'Token':<40} {'Reversible?'}")
print("-" * 90)

tokens_created = []
for u in users[:5]:
    token = tokenizer.tokenize(u["email"], pii_type="email")
    reversed_value = tokenizer.detokenize(token)
    match = "✅ Yes" if reversed_value == u["email"] else "❌ No"
    print(f"  {u['email']:<30} {token:<40} {match}")
    tokens_created.append(token)

# Show idempotency — same email gets same token
print(f"\n💡 Idempotency check:")
same_token = tokenizer.tokenize(users[0]["email"], pii_type="email")
print(f"   Tokenizing '{users[0]['email']}' again → {same_token}")
print(f"   Same as before? {'✅ Yes' if same_token == tokens_created[0] else '❌ No'}")

In [ ]:
# Tokenize an entire user record

def tokenize_user(user, tokenizer):
    """Create a tokenized version of a user record.
    PII fields are replaced with tokens; non-PII fields stay as-is.
    """
    return {
        "id": tokenizer.tokenize(str(user["id"]), "user_id"),
        "name": tokenizer.tokenize(f"{user['first_name']} {user['last_name']}", "name"),
        "email": tokenizer.tokenize(user["email"], "email"),
        # Non-PII fields remain visible for analytics
        "city": user["city"],
        "state": user["state"],
        "account_status": user["account_status"],
        "signup_source": user["signup_source"]
    }

print("📋 Tokenized User Records (safe for analytics teams)")
print("=" * 110)

for u in users[:5]:
    tok = tokenize_user(u, tokenizer)
    print(f"\n  User Token: {tok['id']}")
    print(f"    Name:   {tok['name']}")
    print(f"    Email:  {tok['email']}")
    print(f"    City:   {tok['city']}")
    print(f"    Status: {tok['account_status']}")

print(f"\n💡 Analytics team can work with this data freely.")
print(f"   They can count users per city, analyze signup sources, etc.")
print(f"   But they can NEVER see the real names or emails without Redis access.")

In [ ]:
# GDPR deletion: delete the token mapping to make data truly anonymous

print("🗑️ GDPR Deletion Request Demo")
print("=" * 60)

# User requests account deletion
token_to_delete = tokens_created[0]
original_before = tokenizer.detokenize(token_to_delete)
print(f"\n  Token: {token_to_delete}")
print(f"  Before deletion — resolves to: {original_before}")

# Delete the mapping
deleted = tokenizer.delete_mapping(token_to_delete)
print(f"\n  🗑️ Mapping deleted: {deleted}")

# Try to resolve again
original_after = tokenizer.detokenize(token_to_delete)
print(f"  After deletion — resolves to: {original_after}")

print(f"\n💡 The token still exists in the analytics database,")
print(f"   but it can NEVER be linked back to a real person.")
print(f"   The data is now effectively anonymous — no longer personal data under GDPR.")

## 📊 Comparison: When to Use Each Technique

| Technique | Reversible? | Use When | Trade-off |
|-----------|------------|----------|----------|
| **k-Anonymity** | No | Releasing datasets to researchers | Loses precision (ages become ranges) |
| **l-Diversity** | No | k-anonymity + sensitive attributes | May need to suppress records |
| **Differential Privacy** | No | Aggregate statistics (counts, averages) | Individual queries are noisy |
| **Tokenization** | Yes (with key) | Internal analytics, cross-system joins | Key compromise reveals all PII |

### What Microsoft Uses

- **Windows telemetry**: Differential privacy (ε=1.0) for usage statistics
- **Azure analytics**: Tokenization for cross-service correlation
- **Research datasets**: k-anonymity + l-diversity for published studies
- **LinkedIn**: Differential privacy for salary insights and hiring analytics

### Next Notebook

In **Notebook 4: Data Retention & Purging**, we'll implement policies that automatically delete data when its retention period expires.